In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
# 1. Configurar o Modelo (Aproveitando o teu processador de última geração)
model_po = xgb.XGBRegressor(
    n_jobs=-1, # Usa todos os núcleos disponíveis
    tree_method="hist", # Método de construção de árvores eficiente para grandes datasets
    n_estimators=2000, # Número alto de árvores para permitir mais aprendizado (com early stopping)
    learning_rate=0.01, # Taxa de aprendizado baixa para melhor generalização
    max_depth=8, # Profundidade moderada para capturar relações complexas sem overfitting
    subsample=0.8, # Amostra 80% dos dados para cada árvore para reduzir overfitting
    colsample_bytree=0.8, # Amostra 80% das features para cada árvore para reduzir overfitting
    random_state=42, # Para reprodutibilidade
    eval_metric="rmse", # Métrica de avaliação para regressão
    early_stopping_rounds=50 # Para parar se o modelo não melhorar por 50 rodadas
)

base_dir = Path("data")

# Carregamento correto dos dados de POSTS
X_train_po = pd.read_csv(base_dir / "train" / "X_posts.csv")
y_train_po = pd.read_csv(base_dir / "train" / "y_posts.csv").squeeze("columns")

X_val_po = pd.read_csv(base_dir / "validation" / "X_posts.csv")
y_val_po = pd.read_csv(base_dir / "validation" / "y_posts.csv").squeeze("columns")

X_test_po = pd.read_csv(base_dir / "test" / "X_posts.csv")
y_test_po = pd.read_csv(base_dir / "test" / "y_posts.csv").squeeze("columns")

# Remover colunas de tipo string/object (incompatíveis com XGBoost)
X_train_po = X_train_po.select_dtypes(exclude=['object', 'string'])
X_val_po = X_val_po.select_dtypes(exclude=['object', 'string'])
X_test_po = X_test_po.select_dtypes(exclude=['object', 'string'])

# 2. Treinar (Garante que as variáveis terminam em _po)
print("A treinar modelo de POSTS...")
model_po.fit(
    X_train_po,
    y_train_po,
    eval_set=[(X_val_po, y_val_po)],
    verbose=100
)

# 3. Avaliar no Teste
preds_po = model_po.predict(X_test_po)

print(f"\n=== RESULTADOS POSTS ===")
print(f"MAE: {mean_absolute_error(y_test_po, preds_po):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_po, preds_po)):.2f}")
print(f"R² Score: {r2_score(y_test_po, preds_po):.2f}")